# Clean-read taxonomy and primer evidence for one barcode
Inputs are the parsed SINTAX read table and species/primer summaries from the
batch parser notebook. All statistics use read IDs and accepted confidence >=0.8.
The contamination flag is calculated across PCR plates by the batch analysis,
not independently within this sample. No sequence translation or consensus input.


## Primer naming
The sequence-based dictionary maps original MFT names (keys), such as
`forward_tagged_primer_1` and `reverse_tagged_primer_1`, to PCR names (values),
`bc1002` and `bc1100rc`, respectively. These numbers are not interchangeable.
Only the dictionary values are exported in biological tables: `forward_PCR_primer`
and `reverse_PCR_primer` at read/primer-pair level, and `forward_primers_detected`
and `reverse_primers_detected` in aggregated summaries. Lists contain unique,
semicolon-separated values. The mapping is saved separately as JSON for provenance.


In [ ]:
from pathlib import Path
import os, sys
import pandas as pd
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'code/sintax_analysis.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT / 'code'))
ANALYSIS_DIR = Path(os.environ.get('SINTAX_ANALYSIS_ROOT', str(PROJECT_ROOT / 'data/sintax_analysis')))

import matplotlib.pyplot as plt
SELECTED_BARCODE = os.environ.get('SELECTED_BARCODE', 'barcode01')
FOLDER_DIR = ANALYSIS_DIR / SELECTED_BARCODE
species = pd.read_csv(FOLDER_DIR / f'{SELECTED_BARCODE}_species_summary.tsv', sep='	')
primers = pd.read_csv(FOLDER_DIR / f'{SELECTED_BARCODE}_primer_taxonomy.tsv', sep='	')
qc = pd.read_csv(ANALYSIS_DIR / 'barcode_analysis_summary.tsv', sep='	')
qc = qc[qc.source_folder.eq(SELECTED_BARCODE)]
# Read-level inspection preserves the query ID and original parent-read ID.
read_preview = pd.read_csv(FOLDER_DIR / f'{SELECTED_BARCODE}_reads.tsv', sep='	', nrows=20)
display(qc)
display(read_preview)


In [ ]:
confident_species = species[species.read_count.gt(0)].copy()
display(confident_species)
display(primers.sort_values('read_count', ascending=False).head(40))
contamination_species = confident_species[confident_species.contamination.eq('contamination')]
display(contamination_species)


In [ ]:
figures = FOLDER_DIR / 'figures'
figures.mkdir(exist_ok=True)
plot_data = confident_species.nlargest(25, 'read_count').sort_values('read_count')
fig, ax = plt.subplots(figsize=(11, max(4, .3*len(plot_data))))
colors = plot_data.contamination.map({'contamination':'#c55a45', 'nocontamination':'#3486a0'})
ax.barh(plot_data.species_name, plot_data.read_count, color=colors)
ax.set(xlabel='Cleaned reads with species confidence >=0.8', title=f'{SELECTED_BARCODE}: species identification')
fig.tight_layout()
fig.savefig(figures / f'{SELECTED_BARCODE}_species_read_counts.png', dpi=160)
plt.show()
